<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 6: Databases</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../3.%20application_structure/8.%20blueprints_and_application_factory.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 8: Blueprints →</a>
</div>

# Chapter 7: Database Migrations with Flask-Migrate — Evolving Your Schema

> *"Your database schema will change. The question is whether you'll change it safely or destructively."*

## 🎯 The Big Picture

You built a beautiful `User` model with id, username, and email. Then your product manager asks for a `bio` field. Then `profile_picture_url`. Then `is_email_verified`. Then `last_login`.

Without migrations, every schema change means:
1. Drop the entire database
2. Recreate it from scratch
3. **Lose all your data** — users, posts, everything

That is acceptable in development (you have no real data). It is **catastrophic in production**.

**Migrations** are the solution. They are like **Git commits for your database schema** — each change is recorded, reversible, and can be applied incrementally to any database without losing data.

**What you will learn in this chapter:**
- What migrations are and why they are essential for any real application
- Setting up Flask-Migrate and the `flask db` CLI
- The complete migration workflow: init → edit model → migrate → review → upgrade
- Reading and understanding generated migration scripts
- Rolling back with `flask db downgrade`
- Tracking history with `flask db history` and `flask db current`
- **Alembic's role**: what it is and its relationship to Flask-Migrate
- **The `migrations/` folder structure**: `env.py`, `script.py.mako`, `versions/`
- **Autogenerate limitations**: what Alembic detects vs what it cannot detect
- **The `down_revision` chain**: how Alembic tracks migration history as a linked list
- **Multi-developer workflows** and merge conflict resolution with `flask db merge`
- **Data migrations** using `op.execute()` and `op.batch_alter_table()`
- **Migration naming conventions**: descriptive messages that document schema intent
- Production migration best practices
- Common pitfalls: NOT NULL columns, column renames, data migrations, merge conflicts

> ❓ **Why do I need migrations?** When your data model changes (e.g., you add a column), the database schema must change too. Migrations are version-controlled scripts that apply those changes in order, keeping every developer and your production server in sync.

## 🧠 Core Concepts: The "Why"

### The Git Analogy

Think of database migrations exactly like Git commits for your schema:

```bash
Git:                               Migrations:
─────────────────────────────────────────────────────
git init                     ↔    flask db init
git add . && git commit -m   ↔    flask db migrate -m "description"
git log                      ↔    flask db history
git push (deploy)            ↔    flask db upgrade
git revert HEAD              ↔    flask db downgrade
git checkout <hash>          ↔    flask db downgrade <revision>
```

Each migration file = one recorded, reversible schema change.

### What Happens Without Migrations

```bash
Week 1: Python model has  {id, username, email}
        Database table has {id, username, email}   <- in sync

Week 2: You add to model:  bio = db.Column(db.String(500))
        Python model has  {id, username, email, bio}
        Database table has {id, username, email}   <- MISMATCH!

        $ flask run
        -> User.query.all()
        -> sqlalchemy.exc.OperationalError: no such column: user.bio
```

Migrations keep the Python models and database schema **in sync**.

---

### Alembic and Flask-Migrate: What Is the Difference?

**Alembic** is the underlying migration engine, created by Michael Bayer (the same author as SQLAlchemy). It is completely independent of Flask — you could use Alembic with any Python project, any framework, or no framework at all.

**Flask-Migrate** is a thin wrapper (~200 lines of code) around Alembic that does three things:
- **Integrates Alembic with Flask's application context** — so `flask db` commands can access `current_app`, your `db` instance, and your configuration
- **Exposes Alembic commands as `flask db <command>`** via Flask-CLI — `flask db migrate` calls Alembic's `revision --autogenerate` under the hood
- **Auto-discovers your `db` instance** — so you do not have to manually configure `env.py` to point at your models and database URL

Understanding this distinction matters when you need advanced Alembic features (e.g., multi-head migrations, offline SQL generation, custom `env.py` logic) that Flask-Migrate does not expose directly.

---

### The `down_revision` Chain: How Alembic Tracks History

Every migration file has two key fields at the top:

```python
revision = "a3f8b2c1"       # this file's unique ID (8–12 hex chars, auto-generated)
down_revision = "9b0e1a2c"  # the ID of the migration that must come before this one
```

These fields form a **linked list** — each migration knows exactly which migration came before it. Alembic walks this chain to determine which migrations need to be applied and in what order:

```
None  <-- 9b0e1a2c (initial)  <-- a3f8b2c1 (add_bio)  <-- f7d9e4c2 (add_posts)
                                                                 ^
                                                        "head" = latest migration
```

- `flask db current` — shows where the database currently sits in this chain
- `flask db upgrade` — walks the chain **forward** from current to head
- `flask db downgrade` — walks the chain **backward** by one step (or to a named revision)
- `flask db history` — prints the entire chain

The `alembic_version` table in your database stores the current revision ID so Alembic always knows where you are.

---

### Autogenerate Limitations: What Alembic Can and Cannot Detect

`flask db migrate` (autogenerate) is powerful but not magic. Know its limits before trusting generated migrations:

**Alembic CAN detect:**
- New tables and dropped tables
- New columns and dropped columns
- Column type changes (e.g., `String(100)` to `String(200)`)
- New indexes and dropped indexes
- New unique constraints and dropped unique constraints
- New and removed foreign key constraints

**Alembic CANNOT detect:**
- **Column renames** — it sees a drop + add, not a rename. All data in the renamed column will be lost!
- Changes to Python-side `default=` values (these exist only in the ORM, not in the DB schema)
- Changes to `__tablename__` (it sees drop old table + create new table — losing all data)
- Stored procedures, triggers, or views
- Partial indexes or complex index expressions

**The column rename trap is the most dangerous.** If you rename `User.name` to `User.full_name`, autogenerate produces a migration that drops `name` and creates `full_name` — deleting every user's name in production. Always manually edit the migration to use `op.alter_column` with `new_column_name`:

```python
# What autogenerate may produce (DATA LOSS in production!):
op.drop_column("user", "name")
op.add_column("user", sa.Column("full_name", sa.String(200)))

# What you should write instead (preserves all data):
op.alter_column("user", "name", new_column_name="full_name")
```

---

### Always Review Generated Migrations Before Applying Them

`flask db migrate` is a **starting point**, not a final answer. Before you ever run `flask db upgrade`:

1. Open the generated file in `migrations/versions/`
2. Read every line of `upgrade()` and `downgrade()`
3. Ask: "Does this accurately represent what I intended to change?"
4. Look for missed renames, wrong types, or missing `server_default` values on NOT NULL columns
5. Add any data migration logic needed (e.g., backfilling new columns)

Common reasons to manually edit a generated migration:
- Convert a detected column drop+add into a proper `op.alter_column` rename
- Add a `server_default` for a new NOT NULL column on a non-empty table
- Insert `op.execute()` statements to backfill data
- Remove false-positive diffs (Alembic sometimes detects phantom type differences between databases)

> ❓ **Why use an ORM instead of raw SQL?** An ORM (Object-Relational Mapper) lets you work with database rows as Python objects, so you write Python instead of SQL strings. It also prevents SQL injection by default and makes switching databases straightforward later.

## ⚡ Syntax & First Usage: Setup

Before adding Flask-Migrate, your Flask-SQLAlchemy setup must already be working — you need a `db = SQLAlchemy()` instance and at least one model class. Flask-Migrate wraps that existing setup; it does not replace it.

**What `Migrate(app, db)` does:**
- Registers Flask-Migrate as a Flask extension, adding all `flask db` CLI commands
- Passes your `db.metadata` to Alembic so it knows which models to compare against the live schema during autogenerate
- Sets up the `migrations/` folder as the target for all generated scripts

**Application factory pattern** — if you use `create_app()`, initialize `Migrate` in two steps:

```python
from flask_migrate import Migrate

migrate = Migrate()          # module-level: no app or db yet

def create_app(config=None):
    app = Flask(__name__)
    db.init_app(app)
    migrate.init_app(app, db)  # wire up after db is ready
    return app
```

**Critical ordering rule**: Flask-Migrate must be initialized **after all your model files are imported**. Alembic builds its picture of your schema from `db.metadata`, which is only populated when SQLAlchemy model classes are defined (i.e., when those files are imported). If a model file is never imported before `Migrate(app, db)` runs, Alembic will not know that model exists and will not include it in autogenerate — leading to silently incomplete migrations.

A common pattern is to import all models at the top of the module where you call `Migrate`, even if you do not use them directly there:

```python
from . import models  # ensures all models are registered with db.metadata
migrate.init_app(app, db)
```

In [ ]:
# Step 1: Install
# pip install flask-migrate

# Step 2: Wire into your Flask app
setup_code = '''
from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from flask_migrate import Migrate

app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///blog.db"

db = SQLAlchemy(app)
migrate = Migrate(app, db)   # <- This one line enables all "flask db" commands
'''
print(setup_code)

print()
print("All flask db commands (available after Migrate is initialized):")
print()
commands = [
    ("flask db init",                "Create migrations/ folder. Run ONCE per project."),
    ("flask db migrate -m 'msg'",   "Auto-generate migration script from model changes."),
    ("flask db upgrade",             "Apply all pending migrations to the database."),
    ("flask db downgrade",           "Revert the most recent migration."),
    ("flask db history",             "List all migrations, oldest to newest."),
    ("flask db current",             "Show which migration the database is currently at."),
    ("flask db heads",               "Show the latest migration revision(s)."),
    ("flask db stamp head",          "Mark DB as up-to-date without running migrations."),
    ("flask db show <rev>",          "Show details of a specific migration."),
    ("flask db merge rev1 rev2",     "Create merge migration to resolve parallel migrations."),
]
for cmd, desc in commands:
    print(f"  {cmd:<35}  {desc}")


## 🔬 Deep Dive: The Complete Migration Workflow

The migration workflow has a **strict sequence** — skipping steps or reordering them leads to confusing errors that are hard to debug if you do not understand the underlying model.

**The three most common beginner mistakes:**

1. **Running `flask db upgrade` before `flask db migrate`** — no migration file exists yet; nothing changes, and you are left wondering why your schema is wrong.
2. **Forgetting `flask db upgrade` after `flask db migrate`** — the migration file exists on disk but the database has not been touched; your app crashes with `OperationalError: no such column`.
3. **Editing the generated migration file incorrectly** — e.g., changing `revision` or `down_revision` IDs by hand, which breaks the chain and orphans all subsequent migrations.

**The mental model**: `flask db migrate` writes a **plan** (a Python file in `migrations/versions/`). `flask db upgrade` **executes** that plan against the live database. Both steps are always required when you change a model.

In [ ]:
print("=== THE COMPLETE MIGRATION WORKFLOW ===")
print()

steps = [
    (
        "STEP 0 — Project Initialization (once per project)",
        "$ flask db init",
        [
            "Creates migrations/ folder and alembic.ini configuration",
            "Run this ONCE when starting a new project",
            "Commit the migrations/ folder to git immediately",
        ]
    ),
    (
        "STEP 1 — Generate first migration",
        "$ flask db migrate -m 'initial migration'",
        [
            "Compares Python models vs empty database",
            "Generates a script that creates all your tables",
            "REVIEW the generated file before proceeding!",
        ]
    ),
    (
        "STEP 2 — Apply the migration",
        "$ flask db upgrade",
        [
            "Runs the upgrade() function in the migration script",
            "Creates the actual database tables",
            "Updates the alembic_version table to track current state",
        ]
    ),
    (
        "STEP 3 — Make a model change",
        "Edit models.py: add bio = db.Column(db.String(500))",
        [
            "Add, remove, or modify a column in your Python model",
            "Don't touch the database yet — just the Python file",
        ]
    ),
    (
        "STEP 4 — Generate migration for the change",
        "$ flask db migrate -m 'add bio column to user'",
        [
            "Diffs your models vs current DB schema",
            "Generates an ALTER TABLE script",
            "ALWAYS review the generated file in migrations/versions/",
        ]
    ),
    (
        "STEP 5 — Apply the change",
        "$ flask db upgrade",
        [
            "Runs upgrade(): adds the 'bio' column to the user table",
            "Existing rows get NULL for bio (since nullable=True)",
            "SAFE: no data is lost",
        ]
    ),
]

for title, command, details in steps:
    print(f"  [{title}]")
    print(f"  Command: {command}")
    for d in details:
        print(f"    -> {d}")
    print()


In [ ]:
# The migrations/ folder structure
print("=== migrations/ folder structure ===")
print()
tree = '''
your_project/
    app.py
    models.py
    migrations/                          <- created by "flask db init"
        alembic.ini                      <- Alembic configuration file
        env.py                           <- migration environment setup
        script.py.mako                   <- template for new migration files
        versions/                        <- one .py file per migration
            abc1_initial_migration.py
            def2_add_bio_to_user.py
            ghi3_add_posts_table.py
            jkl4_add_post_tags.py
'''
print(tree)
print("Key rules:")
print("  - Commit the ENTIRE migrations/ folder to git")
print("  - Other developers run 'flask db upgrade' to sync their local DB")
print("  - Production: run 'flask db upgrade' before restarting the app")
print("  - Add *.db to .gitignore (each developer has their own local DB file)")


### Reading a Migration File — Every Line Explained

Alembic migration files are plain Python scripts in `migrations/versions/`. Each file has an `upgrade()` function (apply the change) and a `downgrade()` function (reverse it). Here is a real migration file, annotated line by line:

---

#### The File Structure

Every generated migration contains these sections in order:

| Section | Purpose |
|---------|---------|
| Module docstring | Human-readable summary: revision ID, parent ID, and creation date |
| `from alembic import op` | The Alembic operations object — all schema changes go through `op` |
| `import sqlalchemy as sa` | Type definitions used in column declarations (`sa.String()`, `sa.Integer()`) |
| `revision` | This file's unique ID (8–12 hex characters, randomly generated by Alembic) |
| `down_revision` | The ID of the migration that must be applied before this one (the parent) |
| `branch_labels` | Advanced: labels for parallel migration branches (almost always `None`) |
| `depends_on` | Advanced: explicit dependency on another migration (almost always `None`) |
| `upgrade()` | The forward migration — applies your schema change to the database |
| `downgrade()` | The reverse migration — undoes `upgrade()` exactly |

#### The Two Key Imports

```python
from alembic import op      # op.add_column(), op.create_table(), op.execute(), etc.
import sqlalchemy as sa     # sa.String(), sa.Integer(), sa.Boolean(), sa.ForeignKey()
```

Every schema operation goes through `op`. You never write raw SQL DDL in a migration — though you can use `op.execute()` for DML (INSERT, UPDATE, DELETE) when doing data migrations.

#### The Revision ID System

Alembic generates a random hex string (e.g., `a3f8b2c1d4e5`) as the revision ID each time you run `flask db migrate`. Flask-Migrate names the file using the message you pass with `-m`:

```
migrations/versions/a3f8b2c1d4e5_add_bio_to_user.py
```

Inside the file:

```python
revision = "a3f8b2c1d4e5"   # this migration's identity — do not change this
down_revision = "9b0e1a2c"   # the parent migration — do not change this
```

The chain of `down_revision` pointers forms the complete migration history. Alembic follows this chain to determine the ordered sequence of migrations to apply (forward) or revert (backward).

#### The `env.py` File — Do Not Edit Unless Necessary

Generated once by `flask db init`, `env.py` configures Alembic's runtime environment:
- Reads the database URL from `SQLALCHEMY_DATABASE_URI` via Flask's app context
- Sets `target_metadata = db.metadata` so autogenerate knows what models exist
- Handles transaction management and logging
- Switches between online mode (`flask db upgrade`) and offline mode (`--sql` flag)

Flask-Migrate patches `env.py` automatically to use Flask's application context. You rarely need to edit it. **When you do need custom configuration** — e.g., schema name prefixes for PostgreSQL, custom type renderers, or multiple database support — `env.py` is where those changes live.

#### The `script.py.mako` File

A Mako template used to generate new migration files. Every time you run `flask db migrate`, Alembic renders this template with the new revision ID, detected changes, and timestamp. Edit this file if you want to add standard imports, license headers, or a custom docstring format to all future migrations. Otherwise, leave it as-is.

In [ ]:
# What a generated migration file looks like
# (Triple-double-quotes in the actual file are shown as comments here)
migration_file = '''
# File: migrations/versions/a3f8b2c1d4e5_add_bio_to_user.py
# Generated by: flask db migrate -m "add bio to user"

# Module docstring would normally be here with:
#   Revision ID: a3f8b2c1d4e5
#   Revises: 9b0e1a2c3d4f     <- parent migration (what came before)
#   Create Date: 2024-01-15 10:30:00

from alembic import op           # op = operations (add/drop/alter columns, tables)
import sqlalchemy as sa          # sa = sqlalchemy type definitions

# Unique ID for this migration (used by "flask db current" and downgrade)
revision = "a3f8b2c1d4e5"

# The revision this migration builds on (forms a linked chain)
down_revision = "9b0e1a2c3d4f"

# Required boilerplate for branching support (usually None)
branch_labels = None
depends_on    = None


def upgrade():
    # ── FORWARD: what to do when applying this migration ──────────
    op.add_column(
        "user",                                          # table name
        sa.Column("bio", sa.String(length=500), nullable=True)  # new column
    )
    op.add_column(
        "user",
        sa.Column("last_login", sa.DateTime(), nullable=True)
    )


def downgrade():
    # ── BACKWARD: how to undo this migration ──────────────────────
    # Columns dropped in REVERSE order of addition
    op.drop_column("user", "last_login")
    op.drop_column("user", "bio")
'''
print(migration_file)


In [ ]:
# Common alembic operations in migration files
print("=== alembic operation reference ===")
print()

ops_list = [
    ("Column operations", [
        "op.add_column('user', sa.Column('bio', sa.String(500), nullable=True))",
        "op.drop_column('user', 'bio')",
        "op.alter_column('user', 'bio', nullable=False, type_=sa.Text())",
        "op.alter_column('user', 'old_name', new_column_name='new_name')",
    ]),
    ("Table operations", [
        "op.create_table('tags', sa.Column('id', sa.Integer, primary_key=True), ...)",
        "op.drop_table('tags')",
        "op.rename_table('old_table', 'new_table')",
    ]),
    ("Index operations", [
        "op.create_index('ix_user_email', 'user', ['email'], unique=True)",
        "op.drop_index('ix_user_email', 'user')",
    ]),
    ("Constraint operations", [
        "op.create_unique_constraint('uq_user_email', 'user', ['email'])",
        "op.drop_constraint('uq_user_email', 'user')",
        "op.create_foreign_key('fk_post_user', 'post', 'user', ['user_id'], ['id'])",
    ]),
    ("Data migration (raw SQL)", [
        "conn = op.get_bind()",
        "conn.execute(sa.text('UPDATE user SET role = :r WHERE role IS NULL'), {'r': 'user'})",
        "op.bulk_insert(user_table, [{'username': 'admin', 'email': 'a@b.com'}])",
    ]),
]

for category, examples in ops_list:
    print(f"  {category}:")
    for ex in examples:
        print(f"    {ex}")
    print()


### Rolling Back — `flask db downgrade`

Rolling back undoes the most recent migration by running its `downgrade()` function. This is essential for fixing mistakes quickly in development and for controlled rollbacks in production.

**Downgrade is a safety net, not a routine operation.** Use it deliberately:

- **Before downgrading in production**: always back up the database first. Downgrades that drop columns or tables permanently destroy that data — there is no undo.
- **The `downgrade()` function must be the exact inverse of `upgrade()`**: if `upgrade()` adds a column, `downgrade()` drops it; if `upgrade()` creates a table, `downgrade()` drops it. Alembic generates this automatically — but always verify it makes sense.
- **Some downgrades are inherently lossy**: dropping a column loses its data. When there is no safe rollback path, document this explicitly and prevent accidental execution:

```python
def downgrade():
    # WARNING: This migration drops the 'payment_data' table.
    # Downgrading permanently deletes all payment records.
    # There is no safe rollback — restore from a database backup instead.
    raise NotImplementedError("Downgrade not supported — restore from backup")
```

**Always test the full upgrade → downgrade → upgrade cycle** before pushing to production:

```bash
flask db upgrade      # apply the migration
flask db downgrade    # undo it — verify no errors and no data corruption
flask db upgrade      # re-apply — must behave identically to the first run
```

If the second `flask db upgrade` behaves differently from the first, your `downgrade()` function has a bug.

In [ ]:
rollback_code = '''
# ── Check current state ──────────────────────────────────────────
$ flask db current
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
a3f8b2c1d4e5 (head)     <- currently at this revision

# ── See full history ─────────────────────────────────────────────
$ flask db history
<base>         -> 9b0e1a2c3d4f,  initial migration
9b0e1a2c3d4f   -> a3f8b2c1d4e5 (head),  add bio to user

# ── Downgrade one step (undo most recent migration) ──────────────
$ flask db downgrade
INFO  Running downgrade a3f8b2c1d4e5 -> 9b0e1a2c3d4f...
# Result: "bio" and "last_login" columns REMOVED from user table
# Data in those columns is LOST (this is the point of downgrade)

# ── Downgrade to specific revision ──────────────────────────────
$ flask db downgrade 9b0e1a2c3d4f    # jump to exact revision
$ flask db downgrade base             # revert ALL migrations (empty DB)

# ── Re-apply ─────────────────────────────────────────────────────
$ flask db upgrade                    # re-apply all pending migrations
$ flask db upgrade a3f8b2c1d4e5       # upgrade to specific revision

# ── Step counts ─────────────────────────────────────────────────
$ flask db downgrade -1               # go back exactly 1 step
$ flask db downgrade -3               # go back 3 steps
$ flask db upgrade +2                 # go forward 2 steps
'''
print(rollback_code)


### ⚖️ `db.create_all()` vs Flask-Migrate

Both approaches create tables, but they serve fundamentally different purposes. The table below compares them across the dimensions that matter for real projects:

---

**Hybrid approach for testing**: Use `db.create_all()` in your test setup (for fast in-memory SQLite) **and** Flask-Migrate in development/production. They do not conflict — `db.create_all()` creates tables directly from your models without touching the `alembic_version` table, while Flask-Migrate tracks schema history in that table. Your test database gets a fresh schema every run; your dev/prod database evolves via migrations.

**Stamping an existing database**: If you already have a database (e.g., you have been using `db.create_all()` and want to switch to migrations), do not run `flask db upgrade` — Alembic will try to create tables that already exist and fail. Instead, use:

```bash
flask db stamp head
```

This writes the current `head` revision ID into the `alembic_version` table without running any SQL DDL. The database is now marked as fully up-to-date. Future `flask db upgrade` calls will only apply migrations created after this point.

In [ ]:
comparison_lines = [
    "+------------------------------+--------------------+---------------------------+",
    "| Feature                      | db.create_all()    | Flask-Migrate             |",
    "+------------------------------+--------------------+---------------------------+",
    "| Creates new tables           | Yes (if missing)   | Yes                       |",
    "| Updates existing tables      | NO (ignores them!) | Yes (ALTER TABLE)         |",
    "| Destroys existing data       | No (skips existing)| No (preserves all data)   |",
    "| Tracks schema history        | No                 | Yes (versioned .py files) |",
    "| Rollback support             | No                 | Yes (downgrade command)   |",
    "| Works for team collaboration | Breaks             | Works (commit migrations) |",
    "| Safe for production          | No                 | Yes (standard practice)   |",
    "| Setup time                   | Zero               | 2 minutes                 |",
    "| Use case                     | Tests / greenfield | Every real project        |",
    "+------------------------------+--------------------+---------------------------+",
    "",
    "Rule of thumb:",
    "  db.create_all()  ->  Unit tests (fresh in-memory DB each run)",
    "                   ->  Very first setup of a brand-new project",
    "  Flask-Migrate    ->  Anything with real data or a team",
]
for line in comparison_lines:
    print(line)


### Production Migration Best Practices

**The cardinal rule**: Never run `flask db migrate` in production. Migration generation is a development task — it compares your current models against the database and writes a Python file. Only ever run `flask db upgrade` in production, applying pre-reviewed, version-controlled migration files.

**Standard deployment order** (maintenance-window deployment):
1. Back up the production database
2. Deploy the new code (including the new migration files in `migrations/versions/`)
3. Run `flask db upgrade` — applies any pending migrations
4. Restart the application server

Never restart the app before running the migration — new code referencing columns that do not exist yet causes immediate crashes.

**Zero-downtime deployments** require backward-compatible migrations:
1. Run a migration that is safe with the **old** code (e.g., add a new nullable column)
2. Deploy new code that works with both old schema and new schema
3. Restart — no downtime, no schema mismatch
4. Later: run a second migration to enforce constraints (e.g., make column NOT NULL)

**Multi-developer workflow — handling migration conflicts:**

When Developer A and Developer B both run `flask db migrate` on separate branches, each creates a migration with the same `down_revision`. After both branches are merged to `main`, Alembic sees two heads pointing to the same parent — a **branch** in the migration graph. Running `flask db upgrade` fails with a "multiple head revisions" error.

Detect the conflict:
```bash
flask db heads        # lists all current heads — should be exactly one in a clean repo
```

Resolve with a merge migration:
```bash
flask db merge revision_A revision_B -m "merge_add_bio_and_posts"
```

This creates a new migration file where `down_revision` is a **tuple** containing both heads:
```python
down_revision = ("a3f8b2c1", "f7d9e4c2")
```

After `flask db upgrade`, Alembic applies both branches and then the merge commit, resulting in a single head again.

> **Prevention**: Pull from `main` and run `flask db upgrade` before creating a new migration. Use descriptive names — it is far easier to spot and resolve conflicts when files are named `add_bio_to_user.py` vs `create_posts_table.py` rather than `abc123.py` vs `def456.py`.

In [ ]:
print('''
=== PRODUCTION MIGRATION CHECKLIST ===

BEFORE touching production:
  1. BACKUP the production database!
       PostgreSQL: $ pg_dump mydb > backup_$(date +%Y%m%d_%H%M).sql
       SQLite:     $ cp prod.db prod_backup_$(date +%Y%m%d).db
       MySQL:      $ mysqldump mydb > backup.sql

  2. Test the migration on a STAGING environment first
       $ FLASK_ENV=staging flask db upgrade
       (Catches issues before they hit real users)

  3. Review the migration file manually
       - Does upgrade() do exactly what you intend?
       - Does downgrade() correctly reverse it?
       - Could any step fail halfway through?

SAFE migrations (can apply without downtime):
  + Adding a nullable column
  + Adding a new table
  + Adding an index (use CONCURRENTLY in PostgreSQL)
  + Dropping an unused index

DANGEROUS migrations (require careful planning):
  ! Adding a NOT NULL column without server_default
    -> Will fail if the table has existing rows
    -> Solution: add as nullable, populate data, then add NOT NULL

  ! Dropping a column that code still references
    -> Deploy code change first, THEN drop the column
    -> "Expand/Contract" deployment pattern

  ! Renaming a column
    -> Auto-generate sees DROP + ADD = DATA LOSS!
    -> Use op.alter_column(..., new_column_name=...) explicitly

  ! Large table migrations (millions of rows)
    -> ALTER TABLE locks the table — users see errors
    -> Use pg_repack (PostgreSQL) or pt-online-schema-change (MySQL)

PRODUCTION DEPLOYMENT ORDER:
  $ git pull                   # get new code with migration files
  $ flask db upgrade           # apply migrations to production DB
  $ systemctl restart myapp    # restart the application server
''')


### Data Migrations — When Auto-Generate Isn't Enough

Alembic's `flask db migrate` is excellent at detecting **structural changes** (add/remove columns, create/drop tables, modify types). But it cannot detect **data transformations** — for those, you write migration logic manually.

**When you need a data migration:**
- Splitting one column into two (e.g., `full_name` to `first_name` + `last_name`)
- Populating a new column based on existing data (e.g., deriving `username` from `email`)
- Normalizing data as you introduce a new relationship
- Backfilling a computed default before making a column NOT NULL

**The standard data migration pattern** — three steps in one migration file:
1. Add the new column as **nullable** (existing rows get NULL — always safe)
2. Use `op.execute()` to run UPDATE SQL that backfills data for all existing rows
3. Use `op.alter_column()` to make it NOT NULL (safe now, because all rows have a value)

**`op.execute()` for DML operations** — runs raw SQL within the migration's transaction:
```python
op.execute("UPDATE user SET role = 'user' WHERE role IS NULL")
```
This is the right tool for any INSERT, UPDATE, or DELETE needed during a migration.

**`op.batch_alter_table()` for SQLite** — SQLite does not support `ALTER TABLE` for dropping or modifying columns. Alembic's batch mode works around this by rewriting the entire table: create a temp table with the new schema, copy all data, drop the old table, rename the temp table. Use it for any column modification on SQLite:

```python
with op.batch_alter_table("user") as batch_op:
    batch_op.alter_column("name", new_column_name="full_name")
    batch_op.drop_column("temp_field")
```

**Migration naming conventions** — use descriptive, snake_case messages with `flask db migrate -m`:

```bash
flask db migrate -m "add_bio_to_user"           # clear — tells you what changed
flask db migrate -m "create_posts_table"         # clear — tells you what was added
flask db migrate -m "add_index_on_post_user_id"  # clear — tells you what was optimised
flask db migrate -m "update2"                    # avoid — opens the file to understand
flask db migrate -m "fix"                        # avoid — which fix? in what table?
```

A well-named migration history reads like a changelog of your schema. A poorly-named one forces you to open every file to understand what happened.

In [ ]:
data_migration = '''
# Scenario: Split "full_name" into "first_name" + "last_name"
# Auto-generate cannot do this — it requires manual data transformation

from alembic import op
import sqlalchemy as sa

def upgrade():
    # Step 1: Add new columns as NULLABLE first (safe)
    op.add_column("user", sa.Column("first_name", sa.String(80), nullable=True))
    op.add_column("user", sa.Column("last_name",  sa.String(80), nullable=True))

    # Step 2: DATA MIGRATION — populate new columns from old data
    conn = op.get_bind()
    rows = conn.execute(sa.text("SELECT id, full_name FROM user")).fetchall()

    for row_id, full_name in rows:
        if full_name:
            parts  = (full_name or "").split(" ", 1)
            first  = parts[0]
            last   = parts[1] if len(parts) > 1 else ""
        else:
            first, last = "", ""

        conn.execute(
            sa.text("UPDATE user SET first_name=:f, last_name=:l WHERE id=:id"),
            {"f": first, "l": last, "id": row_id}
        )

    # Step 3: Make NOT NULL now that all rows have values
    op.alter_column("user", "first_name", nullable=False, server_default="")
    op.alter_column("user", "last_name",  nullable=False, server_default="")

    # Step 4: Drop the old column
    op.drop_column("user", "full_name")


def downgrade():
    op.add_column("user", sa.Column("full_name", sa.String(160), nullable=True))
    conn = op.get_bind()
    # Recombine first + last into full_name
    conn.execute(sa.text(
        "UPDATE user SET full_name = first_name || ' ' || last_name"
    ))
    op.alter_column("user", "full_name", nullable=False)
    op.drop_column("user", "last_name")
    op.drop_column("user", "first_name")
'''
print(data_migration)
print()
print("Key insight: Auto-generate handles SCHEMA changes (add/drop/alter).")
print("Data migrations (moving/transforming row data) need op.get_bind() + raw SQL.")


## 🧪 What If? Experimentation

### What If 1: Adding a NOT NULL Column to a Table with Existing Rows?

If you add a non-nullable column without a default value, the database rejects the migration because existing rows would have no value for the new column. Here is the correct approach:

**Why it fails**: When you add `nullable=False` to a column on a table that already has rows, the database must assign a value to every existing row immediately. With no default provided, it cannot — and raises a constraint violation before a single row is migrated.

**Fix 1 — `server_default` in the migration** (simplest for most cases):

```python
# Auto-generate may produce this — fails on non-empty tables:
op.add_column("user", sa.Column("role", sa.String(20), nullable=False))

# Add a server_default so existing rows get a value automatically:
op.add_column("user", sa.Column("role", sa.String(20), nullable=False,
                                 server_default="user"))
```

`server_default` is different from SQLAlchemy's `default=`. It becomes a `DEFAULT` constraint in the database itself (like SQL's `DEFAULT 'user'`), so the DB engine uses it when filling in existing rows during the migration. After the migration runs, you can remove the `server_default` from your model if you want the column to be required at the application level but not have a permanent DB-level default.

**Fix 2 — Three-step approach** (more explicit, better for complex defaults):
1. Add the column as **nullable** (existing rows get `NULL` — always safe)
2. Run `op.execute()` to backfill all existing rows with the desired value
3. Use `op.alter_column()` to add the `NOT NULL` constraint

```python
def upgrade():
    # Step 1: add as nullable — never fails regardless of existing data
    op.add_column("user", sa.Column("role", sa.String(20), nullable=True))
    # Step 2: backfill — give every existing user a default role
    op.execute("UPDATE user SET role = 'user' WHERE role IS NULL")
    # Step 3: now safe to enforce NOT NULL — all rows have a value
    op.alter_column("user", "role", nullable=False)
```

This gives you precise control over what existing rows receive and avoids a permanent DB-level `DEFAULT` you may not want long-term.

In [ ]:
print('''
SCENARIO: You add a NOT NULL column to a table with existing data

  class User(db.Model):
      ...
      # NEW COLUMN with no default, not nullable:
      account_type = db.Column(db.String(20), nullable=False)

  $ flask db migrate -m "add account type"
  # Generated: op.add_column("user", sa.Column("account_type",
  #                           sa.String(20), nullable=False))

  $ flask db upgrade
  # ERROR: Cannot add NOT NULL column with no default value
  # Existing rows cannot have NULL for account_type!

THE FIX — safe three-step approach:

  def upgrade():
      # Step 1: Add as NULLABLE (existing rows get NULL — safe)
      op.add_column("user", sa.Column("account_type", sa.String(20), nullable=True))

      # Step 2: Fill in a value for all existing rows
      conn = op.get_bind()
      conn.execute(sa.text("UPDATE user SET account_type = 'standard'"))

      # Step 3: Now make it NOT NULL (all rows have a value)
      op.alter_column("user", "account_type", nullable=False)

ALTERNATIVE: Use server_default (simpler, let the DB fill the value)
  op.add_column("user", sa.Column(
      "account_type",
      sa.String(20),
      nullable=False,
      server_default="standard"   # database fills existing rows automatically
  ))
  # After upgrade, remove server_default if you don't want it ongoing.

The server_default approach is usually simpler and recommended.
''')


### What If 2: You Accidentally Delete the `migrations/` Folder?

The `migrations/` folder is your schema version history. Deleting it does **not** affect the database itself — the data and structure are still intact — but you lose the ability to manage future changes incrementally, and `flask db` commands stop working.

**What is actually lost:**
- The complete history of schema changes (who changed what, when, and why)
- The ability to `flask db downgrade` to any previous state
- The migration chain (`down_revision` pointers) needed for safe incremental upgrades

**Recovery path:**
1. Run `flask db init` to recreate the `migrations/` scaffolding
2. Run `flask db migrate -m "recreate_schema"` — Alembic diffs against an empty history and generates a migration that creates your entire current schema from scratch
3. **Do not run `flask db upgrade`** — the tables already exist in your database. Instead, run:
   ```bash
   flask db stamp head
   ```
   This records the new migration as already applied, without touching the database.

From this point you have a fresh migration history starting from today's schema state. You lose the incremental history, but you can continue making and applying future migrations safely.

**Prevention**: Commit the entire `migrations/` folder to version control — including `migrations/versions/` and every generated file inside it. Never add `migrations/` to `.gitignore`. These files are source code, not build artefacts.

In [ ]:
print('''
SCENARIO: migrations/ folder is deleted or corrupted

WHAT IS LOST:
  - Complete history of schema changes
  - Ability to downgrade
  - Team members cannot sync their databases
  - CI/CD pipeline breaks

RECOVERY OPTIONS:

Option 1: Restore from git (BEST — this is why we commit migrations/)
  $ git checkout -- migrations/
  The entire migrations/ folder is restored from version control.
  This is why "commit migrations/ to git" is the #1 rule.

Option 2: You have the database but lost the files
  # The database is fine — you just lost the migration history
  $ flask db init                              # recreate the folder

  # Generate a "baseline" migration from current model state
  $ flask db migrate -m "baseline after recovery"
  # Review: this should match your CURRENT schema exactly

  # IMPORTANT: Don't run upgrade (tables already exist)!
  # Instead, stamp the DB as being at this revision:
  $ flask db stamp head
  # This records the revision ID without running any SQL

  # Future "flask db migrate" and "flask db upgrade" work normally

Option 3: Full reset (only if you can lose ALL data)
  $ flask db init
  $ flask db migrate -m "initial migration"
  $ flask db upgrade    # creates tables from scratch (DATA LOST!)

LESSON: Add migrations/ to git in your FIRST commit.
''')


### What If 3: Two Developers Create Migrations at the Same Time?

When two developers each run `flask db migrate` on separate branches and then merge, Alembic's linear history is broken — both migrations point to the same `down_revision`. This is the **migration branch conflict** problem.

**What the conflict looks like:**

```
main:      9b0e1a2c (initial)
                |
branch-a:  a3f8b2c1 (add_bio_to_user)     down_revision = "9b0e1a2c"
branch-b:  f7d9e4c2 (create_posts_table)  down_revision = "9b0e1a2c"
```

After merging both branches to `main`, running `flask db upgrade` fails:
```
ERROR [alembic.util.messaging] Multiple head revisions are present for
given argument 'head'; please specify a specific target revision
```

**Step 1 — Detect the conflict:**
```bash
flask db heads      # lists all current heads — more than one means a conflict
flask db history    # shows the full graph with branching points
```

**Step 2 — Create a merge migration:**
```bash
flask db merge a3f8b2c1 f7d9e4c2 -m "merge_add_bio_and_posts"
```

This generates a new migration file with a tuple `down_revision`:
```python
down_revision = ("a3f8b2c1", "f7d9e4c2")  # both branches are parents
```

**Step 3 — Apply the merge:**
```bash
flask db upgrade    # applies both branches (if not yet applied) and the merge commit
flask db heads      # should now show exactly one head
```

**Prevention tips:**
- Always pull from `main` and run `flask db upgrade` before creating a new migration
- Communicate with your team before running `flask db migrate` on a shared feature
- Name migrations descriptively — `add_bio_to_user.py` vs `create_posts_table.py` makes conflicts obvious during code review
- In larger teams, designate one person per sprint to manage schema changes, or use a dedicated migrations branch

In [ ]:
print('''
SCENARIO: Developer A and Developer B both edit models and run
"flask db migrate" on separate branches simultaneously.

  Branch A creates: revision b1c2d3  (adds bio to User)
                    down_revision = "9b0e1a2c3d4f"

  Branch B creates: revision f5a6b7  (adds Tag model)
                    down_revision = "9b0e1a2c3d4f"  <- SAME parent!

After merging both branches into main:
  The migrations chain now has a FORK (two heads):
  9b0e... -> b1c2... (head 1)
  9b0e... -> f5a6... (head 2)

  $ flask db upgrade
  ERROR: Multiple head revisions!
         Please merge these revisions.

HOW TO RESOLVE:

  # Create a merge migration that unifies the two heads
  $ flask db merge -m "merge bio and tag migrations" b1c2d3 f5a6b7

  # This creates a new migration file:
  #   revision      = "zz_merged"
  #   down_revision = ("b1c2d3", "f5a6b7")   # tuple = merge point
  #
  #   def upgrade(): pass   # merge commit does nothing
  #   def downgrade(): pass

  $ flask db upgrade    # applies all three: b1c2, f5a6, then the merge

  $ flask db history
  9b0e... -> b1c2... (merged)    add bio to user
  9b0e... -> f5a6... (merged)    add tag model
  (b1c2, f5a6) -> zz_merged (head)   merge bio and tag migrations

HOW TO PREVENT:
  - Communicate schema changes with your team before starting
  - Keep schema changes in separate short-lived feature branches
  - Rebase feature branches onto main BEFORE running flask db migrate
  - One team member owns schema changes during a sprint
''')


## 🚀 Real-World Project Link

Every schema change to our **Blog** is a tracked migration. The initial migration creates all four tables (User, Post, Comment, Tag + post_tags association). Adding `last_login` to User is one migration. Adding a `slug` field to Post is another. Adding a full-text search index is yet another. The entire schema evolution is auditable, reversible, and safe to deploy to production without ever losing user data or blog content.

## 📋 Chapter Summary & Cheat Sheet

Everything you need to remember about Flask-Migrate, condensed:

In [ ]:
lines = [
    "# ============================================================",
    "#    CHAPTER 7 CHEAT SHEET — DATABASE MIGRATIONS",
    "# ============================================================",
    "",
    "# SETUP",
    "# pip install flask-migrate",
    "from flask_migrate import Migrate",
    "migrate = Migrate(app, db)    # after: db = SQLAlchemy(app)",
    "",
    "# ONE-TIME INITIALIZATION",
    "# $ flask db init",
    "# Commit the migrations/ folder to git immediately!",
    "",
    "# EVERYDAY WORKFLOW",
    "# 1. Edit your model in models.py",
    '# 2. $ flask db migrate -m "describe the change"',
    "# 3. Review migrations/versions/latest_file.py",
    "# 4. $ flask db upgrade",
    "",
    "# KEY COMMANDS",
    "# flask db init               one-time: create migrations/ folder",
    "# flask db migrate -m 'msg'   generate migration from model diff",
    "# flask db upgrade            apply pending migrations",
    "# flask db downgrade          revert most recent migration",
    "# flask db history            show all migrations",
    "# flask db current            show current database revision",
    "# flask db stamp head         mark DB as up-to-date (no SQL run)",
    "# flask db merge r1 r2        resolve parallel migration heads",
    "",
    "# COMMON OPERATIONS IN MIGRATION FILES",
    "from alembic import op",
    "import sqlalchemy as sa",
    "",
    "# Add column (nullable first for existing data!)",
    "op.add_column('user', sa.Column('bio', sa.String(500), nullable=True))",
    "# Make it NOT NULL after populating data:",
    "op.alter_column('user', 'bio', nullable=False)",
    "",
    "# Drop a column",
    "op.drop_column('user', 'bio')",
    "",
    "# Create a new table",
    "op.create_table('tag', sa.Column('id', sa.Integer, primary_key=True),",
    "                sa.Column('name', sa.String(50), unique=True, nullable=False))",
    "",
    "# Data migration (raw SQL)",
    "conn = op.get_bind()",
    "conn.execute(sa.text('UPDATE user SET role = :r WHERE role IS NULL'), {'r': 'user'})",
    "",
    "# NOT NULL SAFE PATTERN",
    "op.add_column('user', sa.Column('role', sa.String(20), nullable=True))",
    "op.get_bind().execute(sa.text("UPDATE user SET role='user' WHERE role IS NULL"))",
    "op.alter_column('user', 'role', nullable=False)",
]
for line in lines:
    print(line)


### Multi-DB and Environment-Specific Migrations

Larger applications sometimes use multiple databases, or need to apply different migration strategies per environment (e.g. test vs staging vs production). Alembic and Flask-Migrate support this through named binds and environment-specific configuration.

**Environment-specific database URIs** — configure `SQLALCHEMY_DATABASE_URI` per environment using your Flask config classes. Migration commands use whatever URI is active when you run them:

```bash
FLASK_ENV=staging    flask db current   # check revision on staging DB
FLASK_ENV=production flask db current   # check revision on production DB
FLASK_ENV=production flask db upgrade   # apply pending migrations to production
```

Always verify which database you are targeting before running `flask db upgrade` in production.

**Offline migrations** — Alembic can generate SQL scripts instead of connecting to the database directly. Useful when a DBA must review and approve changes before applying to production:

```bash
flask db upgrade --sql > migration.sql   # generate SQL script, do not execute it
# DBA reviews migration.sql, then applies manually via psql or mysql
```

**Named binds for multiple databases** — Flask-SQLAlchemy supports `SQLALCHEMY_BINDS` for models spread across multiple databases. Each bind maintains its own migration stream. Configure `target_metadata` in `env.py` to include metadata from all binds, and use Alembic's branch support to track each database independently.

**Schema naming conventions** (PostgreSQL) — if your tables live in a non-default schema (e.g., `public.user` vs `app.user`), set `include_schemas = True` in `env.py` and configure your models with `__table_args__ = {"schema": "app"}`. Alembic will include schema prefixes in all generated DDL.

In [ ]:
# Handling different environments with migrations
env_config = '''
# config.py — different database URIs per environment
import os

class Config:
    SECRET_KEY = os.environ.get("SECRET_KEY", "dev-secret")
    SQLALCHEMY_TRACK_MODIFICATIONS = False

class DevelopmentConfig(Config):
    SQLALCHEMY_DATABASE_URI = "sqlite:///dev.db"
    DEBUG = True

class TestingConfig(Config):
    SQLALCHEMY_DATABASE_URI = "sqlite:///:memory:"   # in-memory, fast tests
    TESTING = True
    WTF_CSRF_ENABLED = False

class ProductionConfig(Config):
    SQLALCHEMY_DATABASE_URI = os.environ["DATABASE_URL"]   # from env!


# app.py
config_map = {
    "development": DevelopmentConfig,
    "testing":     TestingConfig,
    "production":  ProductionConfig,
}
env = os.environ.get("FLASK_ENV", "development")
app.config.from_object(config_map[env])

# Running migrations per environment:
# $ FLASK_ENV=development flask db upgrade   <- local SQLite
# $ FLASK_ENV=production  flask db upgrade   <- production PostgreSQL

# Flask-Migrate uses the DATABASE_URI from your app config,
# so the same migration files work across all environments.
'''
print(env_config)


### Checking Migration Status in CI/CD

In a CI/CD pipeline, verifying migration consistency before deployment prevents the most common deployment failure: new code running against a database whose schema has not been updated yet.

**Key commands for pipeline integration:**

```bash
# Check whether the database has all migrations applied (exits non-zero if not)
flask db check        # Alembic 1.9+ — use this as a pre-startup health check

# Inspect current revision (useful for logging/debugging in the pipeline)
flask db current

# Apply all pending migrations (always idempotent — safe to run even if up-to-date)
flask db upgrade
```

**Recommended CI/CD migration workflow:**

| Stage | Action |
|-------|--------|
| Pull Request CI | Run `flask db upgrade` on a temp test DB, then the full test suite — catches migration errors before merge |
| Merge to `main` | Deploy code to staging, then `flask db upgrade` on the staging database |
| Production release | Back up first, then `flask db upgrade`, then restart the application |

**Testing the full migration cycle in CI** — always verify that migrations are reversible:

```bash
flask db upgrade      # apply all migrations to test DB
pytest                # run tests against migrated schema
flask db downgrade    # verify downgrade works without errors
flask db upgrade      # verify re-apply works identically
```

**Schema drift detection** — add `flask db check` as a pre-startup check in your deployment script or application initialization. It exits non-zero if there are pending migrations, letting you fail fast before the app starts with a mismatched schema.

In [ ]:
# CI/CD pipeline integration
cicd_code = '''
# In your CI/CD pipeline (GitHub Actions, GitLab CI, etc.):
#
# 1. Run tests with a fresh database:
#    - Use sqlite:///:memory: OR a test PostgreSQL
#    - Run db.create_all() (not migrations) for speed
#
# 2. Check for pending migrations before deploy:
#    flask db check   <- raises error if migrations are pending
#
# 3. Apply migrations as part of deployment:
#    flask db upgrade
#
# GitHub Actions example:
# .github/workflows/deploy.yml:
#
# - name: Run database migrations
#   env:
#     DATABASE_URL: ${{ secrets.DATABASE_URL }}
#   run: |
#     flask db upgrade
#
# - name: Restart app
#   run: |
#     systemctl restart myapp

# Useful migration verification commands:
check_commands = [
    "flask db check       # verify all migrations are applied (exit 1 if not)",
    "flask db current     # show current DB version",
    "flask db heads       # show latest available version",
    "flask db history     # full migration history",
]
for cmd in check_commands:
    print(f"  $ {cmd}")
'''
print(cicd_code)


### A Complete Real-World Migration History

Here is what a realistic sequence of migrations looks like over the life of a project — each entry shows the revision ID, message, and what it represents. This is the exact output format of `flask db history` on a mature application. Reading a well-named history tells you the story of your schema evolution without opening a single migration file:

In [ ]:
# What a real project's migration history looks like
history_example = '''
# $ flask db history (reading oldest to newest)
#
# <base>        -> a1b2c3d4  (initial migration)
#   Created tables: user, post, comment, tag, post_tags
#
# a1b2c3d4      -> b2c3d4e5  (add user profile fields)
#   op.add_column("user", bio)
#   op.add_column("user", profile_picture_url)
#
# b2c3d4e5      -> c3d4e5f6  (add post view counter)
#   op.add_column("post", view_count INTEGER DEFAULT 0)
#
# c3d4e5f6      -> d4e5f6a7  (add email verification)
#   op.add_column("user", is_email_verified BOOLEAN DEFAULT FALSE)
#   op.add_column("user", email_verification_token VARCHAR(64))
#
# d4e5f6a7      -> e5f6a7b8  (add post slugs for SEO URLs)
#   op.add_column("post", slug VARCHAR(200) UNIQUE)
#   op.execute("UPDATE post SET slug = LOWER(REPLACE(title, " ", "-"))")
#   op.alter_column("post", "slug", nullable=False)
#
# e5f6a7b8      -> f6a7b8c9  (add full-text search index)
#   op.create_index("ix_post_body_fts", "post", ["body"], postgresql_using="gin")
#
# f6a7b8c9      -> g7a8b9c0  (add user follower relationship)
#   op.create_table("followers",
#       sa.Column("follower_id", sa.Integer, sa.ForeignKey("user.id")),
#       sa.Column("followed_id", sa.Integer, sa.ForeignKey("user.id")),
#   )
#
# This is 7 migrations. A mature app might have 50-200 migrations.
# Each one is a safe, reversible, documented schema change.
'''
print(history_example)
print()
print("Key observation: migrations grow incrementally.")
print("  Each migration is small and focused on one change.")
print("  This makes them easy to review, test, and roll back.")


### Alembic Configuration Deep Dive

The `migrations/env.py` file controls how Alembic connects to the database and finds your models. Flask-Migrate generates a working `env.py` automatically — you rarely need to edit it. But understanding it lets you customise naming conventions, multi-db setups, and offline migrations.

**Key sections of `env.py`:**

| Section | What it does |
|---------|-------------|
| `target_metadata` | Points to `db.metadata` — Alembic reads this to know what models exist for autogenerate |
| `run_migrations_online()` | Runs when you execute `flask db upgrade/downgrade` — connects to the live DB |
| `run_migrations_offline()` | Runs when you use `--sql` flag — generates SQL without connecting |
| `include_schemas` | Set to `True` to include schema prefixes (for PostgreSQL schemas) |

**Custom naming conventions** — by default, Alembic uses auto-generated names for indexes and constraints, which causes false-positive diffs in autogenerate on some databases. Configure explicit naming conventions in your `db` instance to keep names stable across all databases:

```python
from sqlalchemy import MetaData

convention = {
    "ix": "ix_%(column_0_label)s",
    "uq": "uq_%(table_name)s_%(column_0_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "pk": "pk_%(table_name)s",
}
metadata = MetaData(naming_convention=convention)
db = SQLAlchemy(metadata=metadata)
```

With this, autogenerate produces consistent, readable constraint names across SQLite, PostgreSQL, and MySQL, and eliminates phantom constraint diffs in `flask db migrate`.

In [ ]:
# The migrations/env.py file controls how Alembic connects and runs
env_py_overview = '''
# migrations/env.py (key parts — auto-generated, rarely needs editing)

from flask import current_app

# Get the database URL from your Flask app config (NOT hardcoded)
def get_engine():
    return current_app.extensions["migrate"].db.get_engine()

def get_metadata():
    return current_app.extensions["migrate"].db.metadata

def run_migrations_online():
    connectable = get_engine()
    with connectable.connect() as connection:
        context.configure(
            connection=connection,
            target_metadata=get_metadata(),
            # These control what auto-generate detects:
            compare_type=True,          # detect column type changes
            compare_server_default=True # detect default value changes
        )
        with context.begin_transaction():
            context.run_migrations()

# alembic.ini — minimal config (usually don't edit this)
# [alembic]
# script_location = migrations
# sqlalchemy.url  = (overridden by env.py to use Flask config)
'''
print(env_py_overview)

print()
print("When DOES auto-generate miss changes?")
misses = [
    "Column renames (seen as drop + add = DATA LOSS!)",
    "Table renames",
    "Stored procedures, views, triggers",
    "Some complex constraint changes",
    "Server defaults (only if compare_server_default=True)",
    "Custom types not in SQLAlchemy",
]
for m in misses:
    print(f"  - {m}")
print()
print("Always review generated migrations before running upgrade!")


## 💪 Practice Prompts

**Challenge 1 — Full Migration Lifecycle:**
Start with a `User` model with only `id`, `username`, and `email`. Run `flask db init` then `flask db migrate -m "initial_migration"` then `flask db upgrade`. Now add four new fields: `bio` (String 500, nullable), `profile_picture` (String 300, nullable), `is_verified` (Boolean, default False), and `last_login` (DateTime, nullable). Run `flask db migrate -m "add_profile_fields"`, open the generated file and read it line by line, then run `flask db upgrade`. Test `flask db downgrade` then `flask db upgrade` again. Verify that the second upgrade behaves identically to the first.

**Challenge 2 — NOT NULL Safe Migration:**
Write a migration manually (do not use auto-generate) that adds a `role` column (String 20, NOT NULL) to an existing `User` table that already has rows. Use the three-step approach: (1) add as nullable, (2) `op.execute()` to set all existing rows to `'user'`, (3) `op.alter_column` to add NOT NULL. Verify it applies without error on a table with pre-existing data.

**Challenge 3 — Simulate a Merge Conflict:**
Create two git branches. In branch A, add `bio` to User and generate a migration. In branch B, add a `Post` model and generate a migration. Merge both to `main`. You will have two migration heads. Run `flask db heads` to see both. Use `flask db merge` to create a merge migration. Verify `flask db upgrade` successfully applies all migrations and `flask db heads` shows exactly one head.

**Challenge 4 — Data Migration:**
Add a `full_name` column to `User`, populate it with some data. Then write a migration (manually, not auto-generate) that: (1) adds `first_name` and `last_name` as nullable columns, (2) uses `op.execute()` to split existing `full_name` data into the two new columns using a SQL `SUBSTR` or `INSTR` expression, (3) drops the old `full_name` column using `op.batch_alter_table()` (required for SQLite). Verify the data migration runs correctly and all existing names are preserved in the new columns.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 6: Databases</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../3.%20application_structure/8.%20blueprints_and_application_factory.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 8: Blueprints →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6. databases_with_flask_sqlalchemy.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='../3. application_structure/8. blueprints_and_application_factory.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>